# Every way to specify a systematic with `graphed.vary`

A guided tour, simplest to most complex. Every cell is executed and prints the
`graphed.labels()` / `graphed.points()` / `graphed.variations()` it actually produces — no number
or label below is asserted from memory.

The notebook is self-contained: toy numpy/awkward arrays and a tiny inline correctionlib set built
in the next cell. One level (the process-pool proof) also reads the 50k skim committed in
`data/`, purely for realism.

> **Companion — the full grid on real data.** The [ADL benchmark](./graphed-adl-benchmarks.ipynb) runs the same JES + b-tag setup on the real 50k skim: because the b-tag SF rides the JES-varied jets, one plain `graphed.vary` per nuisance auto-fans to the **full 15-universe grid** (propagation for free, level 9) — the datacard's seven-template union is one `composes_as_union=True` away. This tour is the deep dive into that mechanism: auto-fanout, its prune, its guard, and the off-grid placements (Levels 15-18) that name universes by hand.

## The map

An analyst makes three orthogonal choices, and the tour is their product plus the three ways
universes can relate.

**(a) What is varied.** A bare `Array` (the *loose* form), an event context's **weight**
(`is_weight=True` + `variations=`), or an event context's **collections** (`collections=`). All
three mint the same label grammar `f"{name}_{tag}"` and the same default point `{name: tag}`.

**(b) How many nuisance families.** One call; a second call extending the *same* family; or
independent families, which compose as the **union**, never the cross product.

**(c) How universes relate.** Three genuinely distinct mechanisms, which the word "correlated"
conflates and which this notebook teaches apart:

| Analyst intent | Mechanism | Level |
|---|---|---|
| Two registrations are the same fit parameter | **name identity** — share the nuisance `name` | 8 |
| A correction is a function of a quantity a nuisance moves | **propagation** — compute it from the varied quantity | 9 |
| A variation computed over another nuisance's varied nodes | **auto-fanout** — the joint grid is minted automatically | 10+ |
| A universe named by hand at a prescribed coordinate | **off-grid placement** — re-point a member onto a chosen point | 15+ |

> **The second verb — placement (Levels 15–18).** Everything through Level 14 *declares* members and lets `graphed` derive their universes. Levels 15–18 add *placement*: name a coordinate for a member by hand — the μR×μF 7-point set, prescribed PDF-eigenvector-style directions. Declares and placements share one `variations=` list; the structure of each entry — a `(tag, array)` tuple versus a `{nuisance: coordinate}` map — picks the verb, and misuse raises `graphed.VariationError` (Level 18).

## Vocabulary

- **nuisance** — a family name; the parameter the fit sees.
- **coordinate** — the per-axis displacement (HistFactory's α, combine's ν).
- **point** — a sparse `{nuisance: coordinate}` map. HS3 calls this a *parameter point*.
- **correlated** — shares a nuisance name. Nothing else.
- **propagation** — a correction depends on a quantity a nuisance moves. *Not* correlation.
- **one-at-a-time set** — the axis-aligned unit points; what a datacard normally wants.
- **factorization error** — what a joint universe measures.

**Every universe is a point in nuisance space; a label is a NAME for that point; resolution
projects the requested point onto the axes a container knows and then falls back to nominal.**
`nominal` is the origin: every coordinate at 0.

## Level 0 — setup

Two toy datasets and one toy correctionlib set. Nothing else is imported.

- `ctx0()` — a numpy-backed event context with two collections, `pt` and `eta`. Used wherever the
  level is about *declaration shape* rather than physics.
- `toy_jets()` — jagged jet pT, an awkward array, used where a correction has to be evaluated.
- `TOY_SF` — a correctionlib v2 set whose `up`/`down` uncertainty **grows with pT**
  (2% / 5% / 10% / 20% in the pT bins `[0,30,60,100,∞)`). That pT dependence is one of the two
  sources of the factorization error measured in level 12; a scale factor flat in pT, read on a
  selection frozen at nominal, makes that error read zero — level 12 runs exactly that leg as its
  control.

In [1]:
import json

import awkward as ak
import correctionlib
import numpy as np

import graphed
from graphed import Session
from graphed.awkward import AwkwardBackend, from_awkward, gak
from graphed.context import EventContext
from graphed.numpy import NumpyBackend, from_record


def ctx0():
    """A fresh Session + a 12-event context with `pt` and `eta` collections."""
    s = Session(NumpyBackend())
    r = from_record(s, "ev", pt=np.arange(1.0, 13.0), eta=np.arange(1.0, 13.0) / 10.0)
    return s, EventContext(s, r["pt"], collections={"pt": r["pt"], "eta": r["eta"]})


def toy_jets(n_events=2000, seed=7):
    """Jagged jet pT: 2-5 jets per event, exponential pT, straddling the SF bin edges."""
    rng = np.random.default_rng(seed)
    counts = rng.integers(2, 6, size=n_events)
    pt = rng.exponential(45.0, size=int(counts.sum())) + 10.0
    return ak.unflatten(pt, counts)


def toy_session():
    s = Session(AwkwardBackend())
    return s, from_awkward(s, "jet_pt", toy_jets())


TOY_SF = json.dumps({
    "schema_version": 2,
    "description": "a pT-binned b-tag-like scale factor; the uncertainty grows with pT",
    "corrections": [{
        "name": "toy_sf",
        "version": 1,
        "inputs": [{"name": "systematic", "type": "string"}, {"name": "pt", "type": "real"}],
        "output": {"name": "sf", "type": "real"},
        "data": {
            "nodetype": "category", "input": "systematic",
            "content": [
                {"key": "central", "value": {"nodetype": "binning", "input": "pt",
                    "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                    "content": [1.00, 1.00, 1.00, 1.00], "flow": "clamp"}},
                {"key": "up", "value": {"nodetype": "binning", "input": "pt",
                    "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                    "content": [1.02, 1.05, 1.10, 1.20], "flow": "clamp"}},
                {"key": "down", "value": {"nodetype": "binning", "input": "pt",
                    "edges": [0.0, 30.0, 60.0, 100.0, 10000.0],
                    "content": [0.98, 0.95, 0.90, 0.80], "flow": "clamp"}},
            ],
        },
    }],
}).encode()

SF = correctionlib.CorrectionSet.from_string(TOY_SF.decode())["toy_sf"]


def show(container, *, variations=False):
    """labels(), then points() one per line, then optionally variations()."""
    print("labels    :", graphed.labels(container))
    pts = graphed.points(container)
    print("points    :")
    for label in graphed.labels(container):
        print(f"    {label:22s} {pts[label]}")
    if variations:
        print("variations:", graphed.variations(container))


print("correctionlib", correctionlib.__version__, "| toy SF at pT=25/50/80/200, systematic=up:",
      [round(SF.evaluate("up", p), 2) for p in (25.0, 50.0, 80.0, 200.0)])

correctionlib 2.9.0 | toy SF at pT=25/50/80/200, systematic=up: [1.02, 1.05, 1.1, 1.2]


## Level 1 — one at a time, on a bare array

The simplest systematic: two keyword tags on an `Array`. The label is `f"{name}_{tag}"` and its
point is the axis-aligned `{name: tag}` — every other nuisance sits at 0, which is what absence
means. `nominal` maps to the empty point, the origin.

This is the *loose* form: no event context, no weight, just a varied quantity.

In [2]:
s, c = ctx0()
pt = c["pt"]

jes = graphed.vary(pt, "jes", up=pt * 1.1, down=pt * 0.9)
show(jes)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}


## Level 2 — extending a family

A nuisance family is open. A second `graphed.vary` call with the *same* name adds tags to it, so
the fit still sees **one** parameter. Three tags on one axis, not three axes.

In [3]:
jes2 = graphed.vary(jes, "jes", up2=graphed.nominal(jes) * 1.21)
show(jes2)

labels    : ('nominal', 'jes_up', 'jes_down', 'jes_up2')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    jes_up2                {'jes': 'up2'}


## Level 3 — a weight systematic

`is_weight=True` on an event context: the selection and the observable are identical in every
universe, only the per-event weight moves. `graphed.variations()` reports the kind word
`'weight'`.

In [4]:
s, c = ctx0()
w = c["pt"] * 0.5

btag = graphed.vary(c, "btag", w, is_weight=True,
                    variations={"up": w * 1.2, "down": w * 0.8})
show(btag, variations=True)

labels    : ('nominal', 'btag_up', 'btag_down')
points    :
    nominal                {}
    btag_up                {'btag': 'up'}
    btag_down              {'btag': 'down'}
variations: {'btag': {'up': ('weight', None), 'down': ('weight', None)}}


## Level 4 — a shift systematic

`collections=` instead: the *kinematics* move, so the selection and the observable both change.
Same declaration shape, same label grammar, same default points — but `graphed.variations()`
reports `'shift'`, and downstream every cut is re-evaluated per universe.

In [5]:
s, c = ctx0()
pt = c["pt"]

shift = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
show(shift, variations=True)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
variations: {'jes': {'up': ('shift', None), 'down': ('shift', None)}}


## Level 5 — lockstep: one nuisance, two collections

One nuisance can move several collections **coherently** — a jet-energy scale that must move `pt`
and `eta` together. The universe count does **not** grow: still three labels, because it is still
one axis.

In [6]:
s, c = ctx0()
pt, eta = c["pt"], c["eta"]

lock = graphed.vary(c, "jes", collections={
    "pt":  {"up": pt * 1.1,   "down": pt * 0.9},
    "eta": {"up": eta * 1.01, "down": eta * 0.99},
})
show(lock)

labels    : ('nominal', 'jes_up', 'jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}


## Level 6 — stacked independent families compose as the UNION

Two independent 2-tag families give **5** universes (`1 + 2 + 2`), not 9. A label registered
with a bare declare (no placement) differs from nominal on exactly **one** axis, so no cross product
can arise implicitly. This axis-aligned set is the one-at-a-time set the datacard wants.

In [7]:
ambient = graphed.weight(btag)          # the btag family from level 3
stacked = graphed.vary(btag, "mu", ambient, is_weight=True,
                       variations={"up": ambient * 1.05, "down": ambient * 0.95})
show(stacked, variations=True)
print()
print("2 tags + 2 tags ->", len(graphed.labels(stacked)), "universes, not", 3 * 3)

labels    : ('nominal', 'btag_up', 'btag_down', 'mu_up', 'mu_down')
points    :
    nominal                {}
    btag_up                {'btag': 'up'}
    btag_down              {'btag': 'down'}
    mu_up                  {'mu': 'up'}
    mu_down                {'mu': 'down'}
variations: {'btag': {'up': ('weight', None), 'down': ('weight', None)}, 'mu': {'up': ('weight', None), 'down': ('weight', None)}}

2 tags + 2 tags -> 5 universes, not 9


## Level 7 — shift then weight

A kinematic family and a weight family stack the same way. Note the second line: the **ambient
weight** carries the inherited `jes` labels even though `jes` is not one of *its* registered
families — a container's labels legitimately outrun its own tag map, which is why axis sets are
read from the Session registry rather than derived from the tags.

In [8]:
w7 = shift["pt"] * 0.5                  # the shift context from level 4
both = graphed.vary(shift, "btag", w7, is_weight=True,
                    variations={"up": w7 * 1.2, "down": w7 * 0.8})

print("context labels:", graphed.labels(both))
print("weight  labels:", graphed.labels(graphed.weight(both)))
print("variations    :", graphed.variations(both))

context labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'btag_up__jes_up', 'btag_up__jes_down', 'btag_down__jes_up', 'btag_down__jes_down')
weight  labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'btag_up__jes_up', 'btag_up__jes_down', 'btag_down__jes_up', 'btag_down__jes_down')
variations    : {'btag': {'up': ('weight', None), 'down': ('weight', None)}, 'jes': {'up': ('shift', None), 'down': ('shift', None)}}


## Level 8 — mechanism 1: NAME IDENTITY

The first correlation mechanism. Registering **one nuisance name** as both a shift and a weight
ties them into a single fit parameter — combine's rule verbatim: *"multiple instances of any
nuisance parameter, sharing the same name, are treated as a single parameter."*

The label set does **not** grow: one universe carries both effects. `graphed.variations()` reports
the third kind word `'both'` for the dual tag, while the shift-only tag stays `'shift'`.

In [9]:
s, c = ctx0()
pt = c["pt"]

sh = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
w8 = sh["pt"] * 0.5
dual = graphed.vary(sh, "jes", w8, is_weight=True, variations={"up": w8 * 1.3})

print("labels    :", graphed.labels(dual), "  <- still three; jes_up now moves BOTH")
print("variations:", graphed.variations(dual))

labels    : ('nominal', 'jes_up', 'jes_down')   <- still three; jes_up now moves BOTH
variations: {'jes': {'up': ('both', None), 'down': ('shift', None)}}


## Level 9 — mechanism 2: PROPAGATION

The second mechanism, and the one most often mislabelled "correlation". The toy scale factor is a
**function of jet pT**, and the `jes` nuisance moves pT. Inside the `jes_up` universe the scale
factor must be re-evaluated on the *shifted* jets.

`gak.apply_correction` is container-traversing: hand it the **varied** pT and it returns a `Varied`
over the same labels, with a **distinct node per universe** — the correction genuinely re-evaluated
on each universe's own jets. Nothing declares this; it falls out of passing the varied quantity.

(The older workaround of fanning out by hand over universes is obsolete: `apply_correction`'s plan
now pickles and crosses a process pool, which level 13 proves.)

In [10]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)

sf_central = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                  args=["central", "$0"])

print("type:", type(sf_central).__name__, " labels:", graphed.labels(sf_central))
for label in graphed.labels(sf_central):
    print(f"    {label:10s} -> IR node {graphed.universe(sf_central, label).node_id}")
print()
print("propagation is not correlation: `jes` is the only registered nuisance;")
print("the SF is simply a function of a quantity `jes` moves.")

type: Varied  labels: ('nominal', 'jes_up', 'jes_down')
    nominal    -> IR node 3
    jes_up     -> IR node 4
    jes_down   -> IR node 5

propagation is not correlation: `jes` is the only registered nuisance;
the SF is simply a function of a quantity `jes` moves.


## Level 10 — the joint grid, minted automatically

The third mechanism, and the important one. When a variation is *computed over* another nuisance's
varied nodes, it genuinely **depends on** that axis, and `graphed` mints the full joint grid on its
own — no placement, no manual enumeration.

Below, `sf` is the b-tag-like weight built off the **JES-varied** pT (the propagation of level 9). So
a single plain `graphed.vary(central, "sf", up=…, down=…)` does not give five universes — it gives
**nine**: `nominal`, the two `jes`, the two one-at-a-time `sf`, and the **four joint cross-terms**
`sf_up__jes_up … sf_down__jes_down`.

- the joint label is **machine-minted** as `f"{name}_{tag}__{foreign}"`; you never spell it;
- its **point** names both axes (`{sf: up, jes: up}`) and binds the **real cross node** the graph
  already holds — the b-tag SF evaluated on *that* universe's shifted pT;
- this is the defect the arc removes: pre-fanout the foreign `jes` coordinate was silently collapsed
  to nominal, dropping the cross-term on the floor.

The full grid is the **opinion-free complete spread**. Level 14 shows the three knobs that mold it.

In [11]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0                       # a selection on the SHIFTED pT
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    """Per-jet SF off the VARIED pT (level 9), producted over the selected jets."""
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                   args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


# ONE plain vary -- no placements. central/up/down are each computed over the jes-varied pT, so the sf
# family DEPENDS on the jes axis, and graphed mints the full jes x sf grid automatically.
weight = graphed.vary(sf("central"), "sf",
                      variations={"up": sf("up"), "down": sf("down")})

show(weight)
print()
print(len(graphed.labels(weight)), "universes = 1 nominal + 2 jes + 2 sf + 4 joints.")
print("each joint label is machine-minted 'sf_<tag>__jes_<tag>'; its point names BOTH axes and")
print("binds the real cross node -- the SF on THAT universe's shifted pT, never the nominal one.")

labels    : ('nominal', 'jes_up', 'jes_down', 'sf_up', 'sf_down', 'sf_up__jes_up', 'sf_up__jes_down', 'sf_down__jes_up', 'sf_down__jes_down')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    sf_up                  {'sf': 'up'}
    sf_down                {'sf': 'down'}
    sf_up__jes_up          {'jes': 'up', 'sf': 'up'}
    sf_up__jes_down        {'jes': 'down', 'sf': 'up'}
    sf_down__jes_up        {'jes': 'up', 'sf': 'down'}
    sf_down__jes_down      {'jes': 'down', 'sf': 'down'}

9 universes = 1 nominal + 2 jes + 2 sf + 4 joints.
each joint label is machine-minted 'sf_<tag>__jes_<tag>'; its point names BOTH axes and
binds the real cross node -- the SF on THAT universe's shifted pT, never the nominal one.


## Level 11 — why a joint label executes: resolution by PROJECTION

The auto-minted joints cost almost nothing because they reuse nodes that already exist. A joint label
is asked of containers that know only *some* of its axes; each contributes its member at the
**projection** of the point onto the axes it carries, then falls back to nominal.

Below, the b-tag weight is built over the JES-varied jets, so `graphed.weight(b)` auto-fans to the
`btag × jes` grid. The observable `obs` carries the `jes` axis only:

- `jes_up` → its own shifted member;
- `btag_up` → an axis `obs` does not carry, so **nominal**;
- `btag_up__jes_up` → the point `{btag: up, jes: up}` restricted to `{jes}` is `{jes: up}`, so `obs`
  gets the **shifted** member.

The weight, which carries both axes, returns the registered joint value. This is the whole of the
resolution machinery, and it is why a joint universe costs only the interning of nodes that already
exist.

In [12]:
s, c = ctx0()
pt = c["pt"]
a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
wv = a["pt"] * 0.5                              # an ambient weight off the jes-varied pT
b = graphed.vary(a, "btag", wv, is_weight=True, variations={"up": wv * 1.2, "down": wv * 0.8})
# b's btag members are computed over the jes-varied wv, so weight(b) auto-fans to the btag x jes grid.
weight = graphed.weight(b)
print("weight labels:", graphed.labels(weight))

obs = a["pt"]                                  # knows the `jes` axis ONLY
print()
for label in ("nominal", "jes_up", "btag_up", "btag_up__jes_up"):
    m = graphed.member_of(obs, label)
    print(f"  obs[{label:16s}] node {m.node_id}  {[float(x) for x in list(s.materialize(m))[:3]]}")

print()
print("weight[btag_up__jes_up]:",
      [float(x) for x in list(s.materialize(graphed.universe(weight, 'btag_up__jes_up')))[:3]])

weight labels: ('nominal', 'jes_up', 'jes_down', 'btag_up', 'btag_down', 'btag_up__jes_up', 'btag_up__jes_down', 'btag_down__jes_up', 'btag_down__jes_down')

  obs[nominal         ] node 1  [1.0, 2.0, 3.0]
  obs[jes_up          ] node 3  [1.1, 2.2, 3.3000000000000003]
  obs[btag_up         ] node 1  [1.0, 2.0, 3.0]
  obs[btag_up__jes_up ] node 3  [1.1, 2.2, 3.3000000000000003]

weight[btag_up__jes_up]: [0.66, 1.32, 1.98]


## Level 12 — what a joint universe MEASURES: the factorization error

A joint universe is **not** "covering the correlation" — the fit already correlates by name
(level 8), and HistFactory's response is a *sum of one-dimensional terms*
(`A = nominal + Σ_i I_i(θ_i)`) with no slot for a joint template. What a joint universe measures is
the **error** of that factorization: how far the true two-axis response sits from the linearized
prediction `jes_1D + sf_1D − nominal`.

Three legs, and it is their **ordering** that carries the lesson, not any absolute number:

| leg | SF pT-dependent? | selection on shifted pT? | expected |
|---|---|---|---|
| the teaching case | yes | yes | largest |
| migration only | no | yes | smaller, still nonzero |
| **positive control** | no | no | **0**, to machine precision |

The third leg is the control that proves the instrument reads zero when there is nothing to
measure. A demo built on a pT-*flat* scale factor with a frozen selection reads zero, and a joint
universe built that way would teach the opposite of the intended lesson.

In [13]:
def yields(pt_dependent, live_selection):
    s = Session(AwkwardBackend())
    raw = from_awkward(s, "jet_pt", toy_jets())
    jes_pt = graphed.vary(raw, "jes", up=raw * 1.05, down=raw * 0.95)
    keep = (jes_pt if live_selection else raw) > 30.0
    passes = gak.sum(keep, axis=1) >= 2

    def sf(systematic):
        arg = jes_pt if pt_dependent else (jes_pt * 0.0 + 50.0)   # freeze the SF's pT argument
        per_jet = gak.apply_correction(TOY_SF, "toy_sf", [arg], SF.evaluate,
                                       args=[systematic, "$0"])
        return gak.prod(per_jet[keep], axis=1) * passes

    # plain vary: the four joints sf_<tag>__jes_<tag> are minted by auto-fanout -- no placements.
    w = graphed.vary(sf("central"), "sf",
                     variations={"up": sf("up"), "down": sf("down")})
    return {L: float(ak.sum(s.materialize(graphed.universe(w, L)))) for L in graphed.labels(w)}


def factorization_error(title, tot):
    print(title)
    base = tot["nominal"]
    worst = 0.0
    for jes_label in ("jes_up", "jes_down"):
        for tag in ("up", "down"):
            joint = tot[f"sf_{tag}__{jes_label}"]                  # the auto-minted cross-term
            linear = tot[jes_label] + tot[f"sf_{tag}"] - base      # the fit's 1-D sum
            err = joint - linear
            worst = max(worst, abs(err) / base)
            print(f"    {jes_label:8s} x sf_{tag:5s}: joint={joint:12.6f}  1D-sum={linear:12.6f}"
                  f"  error={err:+11.6f}  ({100 * err / base:+.4f}% of nominal)")
    print(f"    -> largest |error| = {100 * worst:.4f}% of nominal\n")
    return worst


w1 = factorization_error("pT-binned SF, live selection  (the teaching case)", yields(True, True))
w2 = factorization_error("CONTROL  pT-flat SF, live selection  (selection migration only)",
                         yields(False, True))
w3 = factorization_error("CONTROL  pT-flat SF, selection frozen at nominal  (must read 0)",
                         yields(False, False))

print(f"ordering holds: {w1 > w2 > w3}"
      f"   (pT-binned+live {100 * w1:.4f}%  >  migration-only {100 * w2:.4f}%  >  control {w3:.2e})")
print("control is zero to machine precision:", w3 < 1e-9)

pT-binned SF, live selection  (the teaching case)
    jes_up   x sf_up   : joint= 1974.875095  1D-sum= 1942.498495  error= +32.376601  (+2.2130% of nominal)
    jes_up   x sf_down : joint= 1127.367392  1D-sum= 1150.617130  error= -23.249738  (-1.5892% of nominal)
    jes_down x sf_up   : joint= 1828.447268  1D-sum= 1859.498495  error= -31.051226  (-2.1224% of nominal)
    jes_down x sf_down : joint= 1089.826382  1D-sum= 1067.617130  error= +22.209251  (+1.5181% of nominal)
    -> largest |error| = 2.2130% of nominal



CONTROL  pT-flat SF, live selection  (selection migration only)
    jes_up   x sf_up   : joint= 1730.679280  1D-sum= 1721.590460  error=  +9.088820  (+0.6212% of nominal)
    jes_up   x sf_down : joint= 1305.432745  1D-sum= 1313.285928  error=  -7.853183  (-0.5368% of nominal)
    jes_down x sf_up   : joint= 1629.570521  1D-sum= 1638.590460  error=  -9.019939  (-0.6165% of nominal)
    jes_down x sf_down : joint= 1238.010241  1D-sum= 1230.285928  error=  +7.724314  (+0.5280% of nominal)
    -> largest |error| = 0.6212% of nominal

CONTROL  pT-flat SF, selection frozen at nominal  (must read 0)
    jes_up   x sf_up   : joint= 1677.590460  1D-sum= 1677.590460  error=  +0.000000  (+0.0000% of nominal)
    jes_up   x sf_down : joint= 1269.285927  1D-sum= 1269.285928  error=  -0.000000  (-0.0000% of nominal)
    jes_down x sf_up   : joint= 1677.590460  1D-sum= 1677.590460  error=  +0.000000  (+0.0000% of nominal)
    jes_down x sf_down : joint= 1269.285927  1D-sum= 1269.285928  error=  -0.0

## Level 13 — the same program on real data, under a process pool

Two things at once.

**Propagation crosses a process boundary.** `gak.apply_correction` records an External node, and the
plan below pickles and runs on a persistent 4-worker pool — the auto-fanned grid and all — with no
hand fan-out and no closure to ship.

**The datacard control.** The default is the full nine-universe grid; `composes_as_union=True`
collapses it back to the pre-fanout datacard set (nominal, the two `jes`, the two one-at-a-time
`sf`). The four joints are exactly the cross-terms the old collapse dropped silently — here they are
present by default and measurable, and opting out is one keyword, not a hand-built fan-out you can
forget. Always read `graphed.points()` to see which universes a program actually carries.

In [14]:
import uproot

import graphed_histogram as gh
import hist.graphed as hg
from graphed_executors.local import ProcessPoolExecutor


def real_program(compose):
    g = uproot.graphed("data/Run2012B_SingleMu_50k.root:Events", library="ak")
    raw = g.Jet_pt
    jes_pt = graphed.vary(raw, "jes", up=raw * 1.05, down=raw * 0.95)
    keep = jes_pt > 30.0
    passes = gak.sum(keep, axis=1) >= 2

    def sf(systematic):
        per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                       args=[systematic, "$0"])
        return gak.prod(per_jet[keep], axis=1) * passes

    # default: auto-fanout mints the 9-universe grid. compose: collapse back to the datacard union.
    weight = graphed.vary(sf("central"), "sf",
                          variations={"up": sf("up"), "down": sf("down")},
                          composes_as_union=compose)
    h = hg.Hist.new.Reg(1, 0.0, 1e9, name="ht").Double().fill(
        ht=gak.sum(jes_pt[keep], axis=1), weight=[weight])
    return gh.plan({"h": h}, steps_per_file=8, backend="graphed.awkward:AwkwardBackend"), weight


executor = ProcessPoolExecutor(max_workers=4, persistent=True)
try:
    plan, weight = real_program(compose=False)
    res = executor.run(plan)
    tot = {k: float(v.sum()) for k, v in gh.unpack(res.value)["h"].items()}
    print("ProcessPoolExecutor: OK -", res.n_partitions, "partitions, ",
          len(tot), "universes; apply_correction survived the pool")
    print("points['sf_up__jes_up'] =", graphed.points(weight)["sf_up__jes_up"])
    for k, v in tot.items():
        print(f"    {k:16s} {v:14.6f}")
    real = factorization_error("\nfactorization error on the skim", tot)

    plan_c, weight_c = real_program(compose=True)     # composes_as_union -> the datacard union
    tot_c = {k: float(v.sum()) for k, v in gh.unpack(executor.run(plan_c).value)["h"].items()}
    print("CONTROL composes_as_union=True:", len(tot_c), "universes =",
          list(graphed.labels(weight_c)))
    print("   the four joints are gone -- the pre-fanout datacard set. The default kept them, so the")
    print("   factorization error above is measurable rather than dropped on the floor.")
finally:
    executor.close()

ProcessPoolExecutor: OK - 8 partitions,  9 universes; apply_correction survived the pool
points['sf_up__jes_up'] = {'jes': 'up', 'sf': 'up'}
    nominal             9233.000000
    jes_up              9950.000000
    jes_down            8513.000000
    sf_up              11237.497496
    sf_down             7507.938209
    sf_up__jes_up      12156.013417
    sf_up__jes_down    10321.268887
    sf_down__jes_up     8059.085588
    sf_down__jes_down    6949.713250

factorization error on the skim
    jes_up   x sf_up   : joint=12156.013417  1D-sum=11954.497496  error=+201.515921  (+2.1826% of nominal)
    jes_up   x sf_down : joint= 8059.085588  1D-sum= 8224.938209  error=-165.852621  (-1.7963% of nominal)
    jes_down x sf_up   : joint=10321.268887  1D-sum=10517.497496  error=-196.228609  (-2.1253% of nominal)
    jes_down x sf_down : joint= 6949.713250  1D-sum= 6787.938209  error=+161.775041  (+1.7521% of nominal)
    -> largest |error| = 2.1826% of nominal



CONTROL composes_as_union=True: 5 universes = ['nominal', 'jes_up', 'jes_down', 'sf_up', 'sf_down']
   the four joints are gone -- the pre-fanout datacard set. The default kept them, so the
   factorization error above is measurable rather than dropped on the floor.


## Level 14 — controlling the grid: union, prune, and the loud guard

The default fanout is the **opinion-free complete spread**. Three knobs mold it, coarse to fine:

- **`composes_as_union=True`** — collapse to the one-at-a-time datacard union (the pre-fanout
  behaviour), byte-for-byte. This is an *analysis opinion* — "a datacard is one-at-a-time" — applied
  explicitly, not the framework default.
- **a placement** (`variations=[…, {nuisance: coordinate}]`) — prune the grid to a chosen subset of
  coordinate maps (here the diagonal). Over the *dependent* `sf` family it is **reachability-validated**:
  every coordinate must name a universe the grid derives. (Level 15 shows the same syntax re-pointing
  an *independent* member off the grid, and it doubles as the precision knob for custom numeric points.)
- **`max_universes=`** — the loud guard. A default grid whose family sizes multiply past the budget
  (default 64) is refused *before* it is minted, naming the count and the families. `composes_as_union`
  and an explicit placement are never guarded — you asked for exactly those universes.

A dependent *chain* (jes → btag → pu) does not explode to a full cross product: each family fans only
over the shared root axis, so three families give fifteen universes, not twenty-seven. The guard is
there for the genuinely wide grids, and for turning a runaway into a one-line placement selection.

In [15]:
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                   args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


variations = {"up": sf("up"), "down": sf("down")}

# 1. the default -- the full grid, the opinion-free complete spread
full = graphed.vary(sf("central"), "sf", variations=variations)
print("default full grid :", len(graphed.labels(full)), "universes")

# 2. composes_as_union=True -- collapse to the one-at-a-time datacard set (the pre-fanout behaviour)
union = graphed.vary(sf("central"), "sf", variations=variations, composes_as_union=True)
print("composes_as_union :", len(graphed.labels(union)), "universes", list(graphed.labels(union)))

# 3. placements -- mix {nuisance: coordinate} maps into variations= to prune the grid to a chosen
#    subset (here the diagonal). Over the dependent sf family this is reachability-validated.
diag = graphed.vary(sf("central"), "sf",
                    variations=[*variations.items(),
                                {"sf": "up", "jes": "up"}, {"sf": "down", "jes": "down"}])
print("placement prune   :", len(graphed.labels(diag)), "universes",
      [L for L in graphed.labels(diag) if "__" in L])

# 4. max_universes -- the loud guard, fires BEFORE a runaway grid is minted
try:
    graphed.vary(sf("central"), "sf", variations=variations, max_universes=8)
except graphed.GraphedError as e:
    print("max_universes=8   : refused ->", e)

default full grid : 9 universes
composes_as_union : 5 universes ['nominal', 'jes_up', 'jes_down', 'sf_up', 'sf_down']
placement prune   : 7 universes ['sf_up__jes_up', 'sf_down__jes_down']
max_universes=8   : refused -> graphed.vary('sf') would fan out to 9 universes (jes(3) x sf(3)); pass variations= placements to select a subset, or raise max_universes (currently 8)


## Level 15 — placing a member off the grid: the additive re-point

Levels 1–14 use one verb: **declare** — give an array a label, whose point is the axis-aligned default
`{name: tag}` (Level 14's prune only *selects* among points the fanout already derives). The second
verb is **place**: hand `graphed.vary` a `{nuisance: coordinate}` map instead of a `(tag, array)`
tuple, and it names a coordinate for an already-declared label.

Both verbs share one `variations=` list, and the *structure* of each entry picks the verb:

| entry | verb |
|---|---|
| `("a", array)` | **declare** the member `corr_a` |
| `{"corr": "a", "jes": "up", "jer": "up"}` | **place** the label `corr_a` at that point |

What a placement *means* depends on whether the named member genuinely depends on the foreign axes:

- over a member **built from** a varied axis (Level 14's `sf` off the jes-varied jets) it is a
  **prune** — select a joint the auto-fanout already derives, keeping the member's own axis;
- over an **independent** member (read off `graphed.nominal`, carrying no foreign axis) it is an
  **additive re-point** — the label leaves its one-at-a-time default and lands on the prescribed
  foreign-only point, and its **own axis is dropped** (the label is a name for the point, not a cross).

Below, `corr_a` is independent of `jes`/`jer`; the placement re-points it from `{corr: a}` onto
`{jes: up, jer: up}`. A re-point needs a genuinely new **≥2-coordinate** point — a single `{jes: up}`
is already the `jes_up` label's point, and Level 18 shows it refused.

In [16]:
s, c = ctx0()
pt, eta = c["pt"], c["eta"]

# two carrier axes a placement can name: jes on pt, jer on eta
carriers = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
eta = carriers["eta"]
carriers = graphed.vary(carriers, "jer", collections={"eta": {"up": eta * 1.01, "down": eta * 0.99}})

# an INDEPENDENT weight: read off graphed.nominal, so it carries neither jes nor jer
factor = graphed.nominal(carriers["pt"]) * 0.5
reg = graphed.vary(carriers, "corr", factor, is_weight=True,
                   variations=[("a", factor * 3.0),                      # DECLARE corr_a
                               {"corr": "a", "jes": "up", "jer": "up"}])  # PLACE it off-grid

show(graphed.weight(reg))
print()
print("corr_a re-pointed to", graphed.points(graphed.weight(reg))["corr_a"],
      "-- own 'corr' axis dropped; the label is a name for the point.")

labels    : ('nominal', 'jes_up', 'jes_down', 'jer_up', 'jer_down', 'corr_a')
points    :
    nominal                {}
    jes_up                 {'jes': 'up'}
    jes_down               {'jes': 'down'}
    jer_up                 {'jer': 'up'}
    jer_down               {'jer': 'down'}
    corr_a                 {'jer': 'up', 'jes': 'up'}

corr_a re-pointed to {'jer': 'up', 'jes': 'up'} -- own 'corr' axis dropped; the label is a name for the point.


## Level 16 — the μR×μF 7-point scale set

The textbook off-grid case. The renormalisation and factorisation scales, μR and μF, are
**independent** one-at-a-time families (each doubled and halved), but the accepted envelope is the
**7-point set**: the four axis-aligned variations, the nominal, and the two **correlated diagonals**
`(μR, μF) = (2, 2)` and `(0.5, 0.5)` — never the anti-correlated `(2, 0.5)` corners.

The diagonals are exactly two off-grid placements. A `scale` family declares two independent members
`upup`/`dndn`, then places each at the two-coordinate diagonal point over the μR and μF axes. Numeric
coordinates canonicalise to labels: `2 → "2"`, `0.5 → "5em1"` (a filesystem-safe rendering of the
value).

In [17]:
s, c = ctx0()
pt = c["pt"]

# muR and muF: INDEPENDENT weight families with numeric tags -- the 4 axis-aligned scale variations
mu_r = graphed.vary(c, "muR", pt * 0.5, is_weight=True, variations={"2": pt * 0.6, "0.5": pt * 0.4})
ambient = graphed.weight(mu_r)
mu_f = graphed.vary(mu_r, "muF", ambient, is_weight=True,
                    variations={"2": ambient * 1.3, "0.5": ambient * 0.7})

# the two correlated DIAGONALS: declare upup/dndn independent, place each at a (muR, muF) point
base = pt * 0.5
scale = graphed.vary(mu_f, "scale", base, is_weight=True,
                     variations=[("upup", base * 1.5), ("dndn", base * 0.87),
                                 {"scale": "upup", "muR": 2, "muF": 2},
                                 {"scale": "dndn", "muR": 0.5, "muF": 0.5}])

show(graphed.weight(scale))
print()
print(len(graphed.labels(graphed.weight(scale))),
      "universes = nominal + 4 axis-aligned (muR/muF) + 2 correlated diagonals (not the (2, 0.5) corners)")

labels    : ('nominal', 'muR_2', 'muR_5em1', 'muF_2', 'muF_5em1', 'scale_upup', 'scale_dndn')
points    :
    nominal                {}
    muR_2                  {'muR': '2'}
    muR_5em1               {'muR': '5em1'}
    muF_2                  {'muF': '2'}
    muF_5em1               {'muF': '5em1'}
    scale_upup             {'muF': '2', 'muR': '2'}
    scale_dndn             {'muF': '5em1', 'muR': '5em1'}

7 universes = nominal + 4 axis-aligned (muR/muF) + 2 correlated diagonals (not the (2, 0.5) corners)


## Level 17 — prescribed directions, and mixing the two verbs in one call

The diagonal of Level 16 is a **prescribed direction**: a universe an analyst names by hand as a point
over foreign axes. The same shape carries a reparameterised or rotated basis — a **PDF Hessian
eigenvector direction**, a decorrelated JES splitting — wherever the wanted universe is a combination
of parameters rather than one axis's own tag. Below, `corr_off` is placed at `{jes: 1, btag: -1}`, a
two-coordinate point over **two different** families; both coordinates are validated by the same
carrier-reachability walk.

Declares, prunes, and additive re-points all ride **one** `variations=` list and are routed
per-entry, so a single call can prune a *dependent* member's grid **and** re-point an *independent*
member off-grid at once — they act on disjoint members and never contend.

In [18]:
# a prescribed direction over TWO different families: {jes: 1, btag: -1}
s, c = ctx0()
pt = c["pt"]
jes = graphed.vary(c, "jes", pt * 0.5, is_weight=True, variations={"1": pt * 0.6, "-1": pt * 0.4})
w1 = graphed.weight(jes)
btag = graphed.vary(jes, "btag", w1, is_weight=True, variations={"1": w1 * 1.3, "-1": w1 * 0.7})
ambient = graphed.weight(btag)
corr = graphed.vary(btag, "corr", ambient, is_weight=True,
                    variations=[("off", ambient * 1.1), {"corr": "off", "jes": 1, "btag": -1}])
print("prescribed direction   corr_off ->", graphed.points(graphed.weight(corr))["corr_off"],
      "  (1 -> '1', -1 -> 'm1')")

# ONE call: a PRUNE (dependent member) and an ADDITIVE re-point (independent member) together
s, c = ctx0()
pt, eta = c["pt"], c["eta"]
a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
eta = a["eta"]
carriers = graphed.vary(a, "jer", collections={"eta": {"up": eta * 1.01, "down": eta * 0.99}})

dependent = carriers["pt"] * 0.5                     # jes-varied -> the corr family fans over jes
independent = graphed.nominal(carriers["pt"]) * 0.7   # off the nominal -> carries no foreign axis
mixed = graphed.vary(carriers, "corr", dependent, is_weight=True,
                     variations=[("dep", dependent * 1.3), ("ind", independent * 1.1),
                                 {"corr": "dep", "jes": "up"},               # PRUNE: keep this joint
                                 {"corr": "ind", "jes": "up", "jer": "up"}])  # ADDITIVE: re-point
mp = graphed.points(graphed.weight(mixed))
print()
print("prune    corr_dep__jes_up ->", mp["corr_dep__jes_up"], " (own 'corr' axis KEPT -- a real cross)")
print("         corr_dep__jes_down dropped:", "corr_dep__jes_down" not in mp)
print("additive corr_ind         ->", mp["corr_ind"], " (own 'corr' axis DROPPED)")

prescribed direction   corr_off -> {'btag': 'm1', 'jes': '1'}   (1 -> '1', -1 -> 'm1')

prune    corr_dep__jes_up -> {'corr': 'dep', 'jes': 'up'}  (own 'corr' axis KEPT -- a real cross)
         corr_dep__jes_down dropped: True
additive corr_ind         -> {'jer': 'up', 'jes': 'up'}  (own 'corr' axis DROPPED)


## Level 18 — the construction-time refusals, discriminated by `.situation`

Every way `variations=` can be misused raises **`graphed.VariationError`** — a `GraphedError`
subclass whose `.situation` string names which contract broke. So `except graphed.VariationError`
(or the broader `except graphed.GraphedError`) catches them all, and `.situation` tells them apart.
The complete set, each fired below:

| `.situation` | the entry that triggers it |
|---|---|
| `unreachable` | a coordinate that is not a registered tag of its axis (`{jes: sideways}`) |
| `conflict` | a placement together with `composes_as_union=True` — the union throws every joint away |
| `empty` | a placement carrying only its own-name coordinate — the nominal universe already is that |
| `duplicate` | a point already named by another label — a single foreign `{jes: up}` is just `jes_up` |
| `unresolved` | a placement whose own tag was never declared, or a nuisance registered nowhere this call sees |

The `max_universes=` budget is a **separate** guard, not a `VariationError`: a plain `GraphedError`
with no `.situation`. A default grid past the budget is refused *before* it is minted (Level 14),
naming the count and the families so the fix — a placement selection or a raised budget — is obvious.

In [19]:
# ---- refusals over a DEPENDENT grid (Level 14's sf family, off the jes-varied jets) ----
s, jets_pt = toy_session()
jes_pt = graphed.vary(jets_pt, "jes", up=jets_pt * 1.05, down=jets_pt * 0.95)
keep = jes_pt > 30.0
passes = gak.sum(keep, axis=1) >= 2


def sf(systematic):
    per_jet = gak.apply_correction(TOY_SF, "toy_sf", [jes_pt], SF.evaluate,
                                   args=[systematic, "$0"])
    return gak.prod(per_jet[keep], axis=1) * passes


declares = [("up", sf("up")), ("down", sf("down"))]

# unreachable: 'sideways' is not a registered jes tag -- it names no universe the grid derives
try:
    graphed.vary(sf("central"), "sf", variations=[*declares, {"sf": "up", "jes": "sideways"}])
except graphed.VariationError as e:
    print(f"unreachable  .situation={e.situation!r}\n   {e}\n")

# conflict: a placement AND composes_as_union -- the union collapses every joint away
try:
    graphed.vary(sf("central"), "sf", variations=[*declares, {"sf": "up", "jes": "up"}],
                 composes_as_union=True)
except graphed.VariationError as e:
    print(f"conflict     .situation={e.situation!r}\n   {e}\n")


# ---- refusals over an INDEPENDENT member (the additive re-point of Level 15) ----
def repoint(entry):
    """Register an independent corr member on a fresh jes+jer context with one placement entry."""
    s, c = ctx0()
    pt, eta = c["pt"], c["eta"]
    a = graphed.vary(c, "jes", collections={"pt": {"up": pt * 1.1, "down": pt * 0.9}})
    ev = a["eta"]
    car = graphed.vary(a, "jer", collections={"eta": {"up": ev * 1.01, "down": ev * 0.99}})
    factor = graphed.nominal(car["pt"]) * 0.5
    return graphed.vary(car, "corr", factor, is_weight=True,
                        variations=[("a", factor * 3.0), entry])


# empty: a placement carrying only its own coordinate -- the nominal universe already IS that point
try:
    repoint({"corr": "a"})
except graphed.VariationError as e:
    print(f"empty        .situation={e.situation!r}\n   {e}\n")

# duplicate: a single foreign {jes: up} is already the jes_up label's point (a label needs >=2 coords)
try:
    repoint({"corr": "a", "jes": "up"})
except graphed.VariationError as e:
    print(f"duplicate    .situation={e.situation!r}\n   {e}\n")

# unresolved: the placement's own tag 'typo' was never declared
try:
    repoint({"corr": "typo", "jes": "up", "jer": "up"})
except graphed.VariationError as e:
    print(f"unresolved   .situation={e.situation!r}\n   {e}\n")

# the max_universes budget is a SEPARATE guard -- a plain GraphedError, no .situation
try:
    graphed.vary(sf("central"), "sf", variations=dict(declares), max_universes=8)
except graphed.GraphedError as e:
    print(f"over budget  type={type(e).__name__}, has .situation={hasattr(e, 'situation')}\n   {e}")

unreachable  .situation='unreachable'
   unreachable: a placement on graphed.vary('sf'): 'sideways' is not a registered tag of nuisance 'jes', whose tags are ['down', 'up']

conflict     .situation='conflict'
   conflict: graphed.vary('sf') got both composes_as_union=True and a placement; the union collapses every joint away, so there is no joint for a placement to keep

empty        .situation='empty'
   empty: variations= entry {'corr': 'a'} has only the 'corr' coordinate; a foreign coordinate at 0 names the central universe, which is what nominal already is

duplicate    .situation='duplicate'
   duplicate: point {'jes': 'up'} is already registered under label 'jes_up', so label 'corr_a' would be a second name for one universe — two slots, two StrCategory bins and two content hashes

unresolved   .situation='unresolved'
   unresolved: variations= entry {'corr': 'typo', 'jes': 'up', 'jer': 'up'}: 'typo' is not a tag of graphed.vary('corr'), whose tags are ['a']



over budget  type=GraphedError, has .situation=False
   graphed.vary('sf') would fan out to 9 universes (jes(3) x sf(3)); pass variations= placements to select a subset, or raise max_universes (currently 8)


## Where each level lands

| # | Level | What it adds |
|---|---|---|
| 1 | loose up/down | `f"{name}_{tag}"`, default point `{name: tag}` |
| 2 | extending a family | a second call, same nuisance — still one fit parameter |
| 3 | weight (`is_weight=True`) | kind `'weight'`; selection fixed |
| 4 | shift (`collections=`) | kind `'shift'`; selection moves |
| 5 | lockstep | one nuisance, several collections, no extra universes |
| 6 | stacked families | independent families compose as the **union**: 2+2 → 5, not 9 |
| 7 | shift then weight | the ambient weight carries inherited labels |
| 8 | **name identity** | one name, both effects, one universe; kind `'both'` |
| 9 | **propagation** | `gak.apply_correction` over a `Varied`, one node per universe |
| 10 | **auto-fanout** | a dependent variation mints the full joint grid — no placement |
| 11 | projection | how a joint label resolves on a container that knows one axis |
| 12 | factorization error | what a joint universe measures, with a zero control |
| 13 | real data + process pool | the grid crosses the pool; `composes_as_union` is the datacard control |
| 14 | controlling the grid | `composes_as_union` / placement prune / `max_universes` guard |
| 15 | **off-grid placement** | a placement over an *independent* member re-points it off the grid; own axis dropped |
| 16 | μR×μF 7-point set | four axis-aligned + two correlated diagonals, minted as placements |
| 17 | prescribed directions + mixing | a reparameterised direction over two families; prune and re-point in one call |
| 18 | refusals | `VariationError.situation`: unreachable / conflict / empty / duplicate / unresolved (+ the `max_universes` guard) |

## Traps worth carrying away

- **A dependency mints the grid; independence does not.** A variation computed over another
  nuisance's varied nodes auto-fans (level 10); two independent axes compose as the union (level 6).
  If you expected joints and got a union, the member was not actually built over the varied nodes.
- **`composes_as_union=True` is an analysis opinion, applied explicitly.** The default is the full
  grid — the opinion-free complete spread. A datacard's one-at-a-time set is one keyword away.
- **A placement's meaning follows the member's dependence.** A `{nuisance: coordinate}` map in
  `variations=` *prunes* a dependent member's auto-grid (own axis kept, coordinates on-grid) and
  *re-points* an independent member off the grid (own axis dropped, any reachable coordinate). The
  structure of the entry — tuple versus map — picks declare versus place; there is no separate
  placement parameter.
- **`graphed.points()` is the ground truth.** It reports the coordinate map behind every label;
  read it before trusting a joint number. On executed results it raises — points are a record-time
  fact, not carried on disk.
- **The guard fires before a runaway is minted.** `max_universes` names the count and the families;
  turn the runaway into a one-line placement selection.